# Model 3 — Simplified EEG CNN + MLP (NaN-safe, Hackathon Edition)
**Multimodal Intraoperative Prediction System — Model 3 of 5**

Simplified architecture replacing WaveNet + Transformer with:
- **1D CNN encoder** (3 conv layers with residual-style skip) — stable, no vanishing/exploding gradients
- **MLP fusion** of CNN embeddings + BIS/SQI spectral features
- Single supervised stage (no MSM pretraining) — fast, hackathon-friendly
- Robust NaN filling: interpolate → forward-fill → backward-fill
- Outputs: IOH labels (5/10/15 min) + N/H label (3-class)
- EEG sample rate: 128 Hz, window: 30 s

In [ ]:
from __future__ import annotations
import gc, logging, math, os, random, time
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import stft as scipy_stft
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import vitaldb

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
CONFIG: Dict = {
    # Identity
    "model_name":   "model_3_simple",
    "log_file":     "model_3_simple.log",

    # VitalDB tracks — raw EEG at 128 Hz
    "wav_tracks":    ["BIS/EEG1_WAV", "BIS/EEG2_WAV"],
    "wav_interval":  1 / 128,

    # BIS scalar channels at 1 Hz (only what we use)
    "scalar_tracks": [
        "BIS/BIS",           # 0
        "BIS/SQI",           # 1
        "Solar8000/ART_MBP", # 2  — IOH label
        "Solar8000/HR",      # 3  — N/H label
    ],
    "scalar_interval": 1,

    # Clinical metadata
    "clinical_params": ["anestart", "aneend", "opstart", "opend", "ane_type"],

    # Case eligibility
    "min_case_duration_sec": 1800,
    "min_sqi_coverage":      0.70,
    "sqi_train_threshold":   70,

    # Window / stride
    "history_sec":    1800,   # 30-min history at 1 Hz
    "stride_sec":     60,
    "max_nan_frac":   0.20,
    "max_fill_gap_sec": 60,

    # IOH label
    "ioh_thresholds_mmhg": 65.0,
    "ioh_min_duration_sec": 60,
    "ioh_horizons_sec": [300, 600, 900],

    # N/H label
    "nh_n_classes":      3,
    "nh_label_smoothing": 0.10,
    "nh_hr_high":        90,
    "nh_bis_low":        60,
    "nh_bis_high":       65,
    "nh_hr_rise_frac":   0.15,

    # EEG waveform
    "eeg_fs":               128,
    "eeg_window_sec":       30,
    "eeg_samples_per_step": 3840,   # 30 s × 128 Hz
    "eeg_clip_uv":          500.0,

    # STFT spectral features (computed per 1-Hz step)
    "stft_win_sec": 4,
    "stft_overlap": 0.5,
    "stft_bands": {
        "delta": (0.5, 4.0),
        "theta": (4.0, 8.0),
        "alpha": (8.0, 13.0),
        "beta":  (13.0, 30.0),
    },
    "bsr_window_sec":    63,
    "bsr_amplitude_uv":  5.0,
    "n_spectral_features": 6,   # 4 band powers + SEF95 + BSR

    # ── Simplified CNN encoder ────────────────────────────────────────────────
    # Input: (B, 2, 3840)  →  (B, cnn_out_dim)
    "cnn_channels":    [32, 64, 128],   # three conv blocks, progressively wider
    "cnn_kernel_size": 7,
    "cnn_out_dim":     128,

    # ── MLP fusion ────────────────────────────────────────────────────────────
    # Fuses: cnn_out_dim + n_scalar_summary + n_spectral_summary
    # scalar summary = mean + std of BIS and SQI over history → 4 values
    # spectral summary = mean of 6 spectral features over history → 6 values
    "scalar_summary_dim":   4,
    "spectral_summary_dim": 6,
    "mlp_hidden_dims": [256, 128],
    "mlp_dropout":     0.3,
    "embedding_dim":   128,

    # Supervised training
    "sup_epochs":                50,
    "sup_lr":                    1e-3,
    "batch_size":                16,
    "gradient_accumulation_steps": 2,
    "early_stopping_patience":   8,
    "weight_decay":              1e-4,
    "max_grad_norm":             1.0,
    "warmup_epochs":             2,

    # Data split
    "val_frac":  0.10,
    "test_frac": 0.10,
    "seed":      42,

    # Output files
    "shap_output":    "shap_model_3_simple.png",
    "pred_output":    "prediction_model_3_simple.png",
    "checkpoint_sup": "ckpt_model_3_simple.pt",
}


In [ ]:
def setup_logging(log_file: str) -> logging.Logger:
    fmt    = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s",
                                datefmt="%H:%M:%S")
    logger = logging.getLogger("model_3_simple")
    logger.setLevel(logging.DEBUG)
    if not logger.handlers:
        ch = logging.StreamHandler(); ch.setLevel(logging.INFO); ch.setFormatter(fmt)
        fh = logging.FileHandler(log_file, mode="w"); fh.setLevel(logging.DEBUG); fh.setFormatter(fmt)
        logger.addHandler(ch); logger.addHandler(fh)
    return logger

log = setup_logging(CONFIG["log_file"])


In [ ]:
def load_clinical_metadata(config: Dict) -> pd.DataFrame:
    log.info("Loading clinical metadata …")
    caseids = list(range(1, 6389))
    df = pd.read_csv("clinical_data.csv")
    df = df[df["caseid"].isin(caseids)]
    cols = ["caseid"] + config["clinical_params"]
    df = df[[c for c in cols if c in df.columns]]
    log.info(f"Clinical metadata: {len(df)} rows")
    return df


In [ ]:
def load_waveform_data(
    caseids: List[int],
    config: Dict,
) -> Tuple[Dict[int, np.ndarray], Dict[int, np.ndarray]]:
    wav_data: Dict[int, np.ndarray] = {}
    scalar_data: Dict[int, np.ndarray] = {}

    for caseid in tqdm(caseids, desc="Loading EEG waveforms"):
        try:
            arr_wav = vitaldb.load_case(caseid, config["wav_tracks"],
                                         interval=config["wav_interval"])
        except Exception as e:
            log.warning(f"caseid {caseid}: wav failed — {e}"); continue
        if arr_wav is None: continue

        try:
            arr_scalar = vitaldb.load_case(caseid, config["scalar_tracks"],
                                            interval=config["scalar_interval"])
        except Exception as e:
            log.warning(f"caseid {caseid}: scalar failed — {e}"); del arr_wav; gc.collect(); continue
        if arr_scalar is None: del arr_wav; gc.collect(); continue

        wav_data[caseid]    = arr_wav.astype(np.float32)
        scalar_data[caseid] = arr_scalar.astype(np.float32)

    log.info(f"Loaded {len(wav_data)} cases")
    return wav_data, scalar_data


In [ ]:
def fill_nan_series(arr: np.ndarray, max_gap_sec: int) -> np.ndarray:
    """Fill NaN with: interpolate (within gap limit) → forward-fill → backward-fill.
    Operates column-wise on 2D array (T, F).
    """
    out = arr.copy()
    T, F = out.shape
    for f in range(F):
        s = pd.Series(out[:, f])
        # 1. Interpolate short gaps (linear)
        s = s.interpolate(method="linear", limit=max_gap_sec, limit_direction="both")
        # 2. Forward-fill any remaining NaN
        s = s.ffill()
        # 3. Backward-fill leading NaN
        s = s.bfill()
        out[:, f] = s.values.astype(np.float32)
    return out


In [ ]:
def remove_outliers(arr: np.ndarray) -> np.ndarray:
    """Clip physiologically implausible values.
    Column order matches CONFIG['scalar_tracks']:
      0:BIS/BIS  1:BIS/SQI  2:ART_MBP  3:HR
    """
    out = arr.copy()
    bounds = [(0,100), (0,100), (20,200), (20,250)]
    for i, (lo, hi) in enumerate(bounds[:out.shape[1]]):
        col = out[:, i]
        out[:, i] = np.where((col < lo) | (col > hi), np.nan, col)
    return out


In [ ]:
def compute_spectral_features(eeg_window: np.ndarray, config: Dict) -> np.ndarray:
    """Compute 6 spectral features from a 30-s EEG segment at 128 Hz.
    Returns [delta, theta, alpha, beta, sef95, bsr] — log1p scaled.
    """
    fs      = config["eeg_fs"]
    bands   = config["stft_bands"]
    bsr_amp = config["bsr_amplitude_uv"]

    nperseg  = min(int(config["stft_win_sec"] * fs), len(eeg_window))
    nperseg  = max(nperseg, 8)
    noverlap = int(nperseg * config["stft_overlap"])

    try:
        freqs, _, Zxx = scipy_stft(eeg_window, fs=fs, window="hann",
                                    nperseg=nperseg, noverlap=noverlap)
        mean_power = (np.abs(Zxx) ** 2).mean(axis=1)
        band_powers = []
        for (flo, fhi) in bands.values():
            mask = (freqs >= flo) & (freqs < fhi)
            bp   = mean_power[mask].sum() if mask.any() else 0.0
            band_powers.append(float(np.log1p(bp)))
        total = mean_power.sum()
        if total > 0:
            cum = np.cumsum(mean_power)
            idx = np.searchsorted(cum, 0.95 * total)
            sef95 = float(freqs[min(idx, len(freqs)-1)])
        else:
            sef95 = 0.0
    except Exception:
        band_powers = [0.0, 0.0, 0.0, 0.0]
        sef95 = 0.0

    bsr_samples = int(config["bsr_window_sec"] * fs)
    seg = eeg_window[-bsr_samples:] if len(eeg_window) >= bsr_samples else eeg_window
    bsr = float((np.abs(seg) < bsr_amp).mean()) if len(seg) > 0 else 0.0
    return np.array(band_powers + [sef95, bsr], dtype=np.float32)


In [ ]:
def max_run_length(arr: np.ndarray) -> int:
    if arr.sum() == 0: return 0
    mx = cur = 0
    for v in arr:
        cur = (cur + 1) if v else 0
        mx  = max(mx, cur)
    return mx


def build_ioh_labels(map_series: np.ndarray, config: Dict) -> Dict[str, np.ndarray]:
    threshold = config["ioh_thresholds_mmhg"]
    min_dur   = config["ioh_min_duration_sec"]
    labels: Dict[str, np.ndarray] = {}
    n = len(map_series)
    for horizon_sec in config["ioh_horizons_sec"]:
        lbl = np.zeros(n, dtype=np.float32)
        for t in range(n - horizon_sec):
            window = map_series[t: t + horizon_sec]
            if max_run_length((window < threshold).astype(int)) >= min_dur:
                lbl[t] = 1.0
        labels[f"ioh_{horizon_sec // 60}"] = lbl
    return labels


def build_nh_labels(bis_series: np.ndarray, hr_series: np.ndarray,
                    config: Dict) -> np.ndarray:
    n        = len(bis_series)
    labels   = np.zeros(n, dtype=np.int64)
    lookback = 300
    for t in range(lookback, n):
        bis_t = bis_series[t];  hr_t = hr_series[t]
        if np.isnan(bis_t) or np.isnan(hr_t): continue
        if hr_t > config["nh_hr_high"] and bis_t < config["nh_bis_low"]:
            labels[t] = 1; continue
        hr_past = hr_series[t - lookback: t]
        if not np.all(np.isnan(hr_past)):
            hr_mean = float(np.nanmean(hr_past))
            if hr_mean > 0 and (hr_t - hr_mean) / hr_mean > config["nh_hr_rise_frac"]:
                if bis_t > config["nh_bis_high"]:
                    labels[t] = 2
    return labels


In [ ]:
def generate_windows(
    wav_data:    Dict[int, np.ndarray],
    scalar_data: Dict[int, np.ndarray],
    clinical_df: pd.DataFrame,
    config:      Dict,
) -> List[Dict]:
    """
    Each window dict contains:
      - scalar_summary : (4,)   mean/std of BIS + SQI over history window
      - spectral_summary: (6,)  mean spectral features over history window
      - win_wav_flat   : (3840*2,) last 30-s EEG both channels, z-scored
      - label_ioh_5/10/15, label_nh
    """
    windows: List[Dict] = []
    history_sec          = config["history_sec"]
    stride_sec           = config["stride_sec"]
    max_nan_frac         = config["max_nan_frac"]
    eeg_samples_per_step = config["eeg_samples_per_step"]
    eeg_fs               = config["eeg_fs"]

    BIS_COL = 0; SQI_COL = 1; MAP_COL = 2; HR_COL = 3

    clinical_df = clinical_df.set_index("caseid")

    for caseid, arr_scalar in tqdm(scalar_data.items(), desc="Generating windows"):
        arr_wav = wav_data.get(caseid)
        if arr_wav is None: continue
        if caseid not in clinical_df.index: del arr_wav; gc.collect(); continue

        T_scalar = arr_scalar.shape[0]
        if T_scalar < config["min_case_duration_sec"]:
            del arr_wav; gc.collect(); continue

        # SQI coverage check
        sqi_col   = arr_scalar[:, SQI_COL]
        valid_sqi = ~np.isnan(sqi_col)
        if valid_sqi.sum() == 0 or (sqi_col[valid_sqi] >= config["sqi_train_threshold"]).mean() < config["min_sqi_coverage"]:
            del arr_wav; gc.collect(); continue

        # Pre-process scalar
        arr_scalar = remove_outliers(arr_scalar)
        arr_scalar = fill_nan_series(arr_scalar, config["max_fill_gap_sec"])

        # Labels
        ioh_labels = build_ioh_labels(arr_scalar[:, MAP_COL], config)
        nh_labels  = build_nh_labels(arr_scalar[:, BIS_COL], arr_scalar[:, HR_COL], config)

        has_any_ioh = any(lbl.sum() > 0 or (1 - lbl).sum() > 0 for lbl in ioh_labels.values())
        if not has_any_ioh: del arr_wav; gc.collect(); continue

        # Spectral features per 1-Hz timestep (EEG channel 0 only, light)
        eeg_ch0 = arr_wav[:, 0]
        spectral_features = np.zeros((T_scalar, 6), dtype=np.float32)
        for t in range(T_scalar):
            wav_end   = int((t + 1) * eeg_fs)
            wav_start = max(0, wav_end - eeg_samples_per_step)
            if wav_end > len(eeg_ch0): break
            seg = eeg_ch0[wav_start:wav_end]
            spectral_features[t] = compute_spectral_features(seg, config)

        for t_end in range(history_sec, T_scalar, stride_sec):
            t_start    = t_end - history_sec
            win_scalar = arr_scalar[t_start:t_end, :]   # (1800, 4)

            # NaN guard — any channel over threshold → skip
            if any(np.isnan(win_scalar[:, f]).mean() > max_nan_frac for f in range(win_scalar.shape[1])):
                continue

            win_scalar = np.nan_to_num(win_scalar, nan=0.0)

            # Raw EEG last 30 s → flat vector (3840*2)
            wav_end   = int(t_end * eeg_fs)
            wav_start = max(0, wav_end - eeg_samples_per_step)
            if wav_end > arr_wav.shape[0]: continue
            win_wav = arr_wav[wav_start:wav_end, :]   # (3840, 2)
            if win_wav.shape[0] < eeg_samples_per_step: continue
            win_wav = np.nan_to_num(win_wav, nan=0.0)
            win_wav = np.clip(win_wav, -config["eeg_clip_uv"], config["eeg_clip_uv"])

            # Compact summaries for MLP (avoids huge tensors)
            bis_win = win_scalar[:, BIS_COL]
            sqi_win = win_scalar[:, SQI_COL]
            scalar_summary = np.array([
                bis_win.mean(), bis_win.std(),
                sqi_win.mean(), sqi_win.std(),
            ], dtype=np.float32)

            spectral_summary = spectral_features[t_start:t_end, :].mean(axis=0)  # (6,)

            sqi_at_end = arr_scalar[t_end - 1, SQI_COL]

            label_ioh_5  = float(ioh_labels["ioh_5"][t_end])  if t_end < len(ioh_labels["ioh_5"])  else 0.0
            label_ioh_10 = float(ioh_labels["ioh_10"][t_end]) if t_end < len(ioh_labels["ioh_10"]) else 0.0
            label_ioh_15 = float(ioh_labels["ioh_15"][t_end]) if t_end < len(ioh_labels["ioh_15"]) else 0.0
            label_nh     = int(nh_labels[t_end]) if t_end < len(nh_labels) else 0

            windows.append({
                "caseid":           caseid,
                "window_start":     t_start,
                "win_wav":          win_wav.astype(np.float32),        # (3840, 2)
                "scalar_summary":   scalar_summary,                    # (4,)
                "spectral_summary": spectral_summary,                  # (6,)
                "label_ioh_5":      label_ioh_5,
                "label_ioh_10":     label_ioh_10,
                "label_ioh_15":     label_ioh_15,
                "label_nh":         label_nh,
                "sqi_end":          float(sqi_at_end) if not np.isnan(sqi_at_end) else 0.0,
            })

        del arr_wav
        if caseid in wav_data: del wav_data[caseid]
        gc.collect()

    log.info(f"Total windows generated: {len(windows)}")
    return windows


In [ ]:
def fit_scalers(train_windows: List[Dict]):
    """Fit StandardScalers for scalar_summary, spectral_summary, and EEG waveform."""
    scalar_stack   = np.stack([w["scalar_summary"]   for w in train_windows])  # (N, 4)
    spectral_stack = np.stack([w["spectral_summary"] for w in train_windows])  # (N, 6)
    wav_stack      = np.concatenate([w["win_wav"].reshape(-1, 2) for w in train_windows], axis=0)

    sc_scalar   = StandardScaler().fit(scalar_stack)
    sc_spectral = StandardScaler().fit(spectral_stack)
    eeg_mean    = wav_stack.mean(axis=0)
    eeg_std     = wav_stack.std(axis=0).clip(min=1e-6)
    log.info("Scalers fitted.")
    return sc_scalar, sc_spectral, eeg_mean, eeg_std


def apply_scalers(windows: List[Dict], sc_scalar, sc_spectral, eeg_mean, eeg_std) -> List[Dict]:
    for w in windows:
        w["scalar_summary"]   = sc_scalar.transform(w["scalar_summary"][None])[0].astype(np.float32)
        w["spectral_summary"] = sc_spectral.transform(w["spectral_summary"][None])[0].astype(np.float32)
        w["win_wav"]          = ((w["win_wav"] - eeg_mean) / eeg_std).astype(np.float32)
    return windows


In [ ]:
def split_cases(all_caseids, val_frac=0.10, test_frac=0.10, seed=42):
    train_val, test = train_test_split(all_caseids, test_size=test_frac, random_state=seed)
    val_size        = val_frac / (1.0 - test_frac)
    train, val      = train_test_split(train_val, test_size=val_size, random_state=seed)
    log.info(f"Split: train={len(train)} val={len(val)} test={len(test)} cases")
    return train, val, test


In [ ]:
class EEGDataset(Dataset):
    def __init__(self, windows: List[Dict]) -> None:
        self.windows = windows

    def __len__(self) -> int:
        return len(self.windows)

    def __getitem__(self, idx: int) -> Dict:
        w = self.windows[idx]
        return {
            "wav":          torch.FloatTensor(w["win_wav"]),               # (3840, 2)
            "scalar_sum":   torch.FloatTensor(w["scalar_summary"]),        # (4,)
            "spectral_sum": torch.FloatTensor(w["spectral_summary"]),      # (6,)
            "label_ioh_5":  torch.FloatTensor([w["label_ioh_5"]]),
            "label_ioh_10": torch.FloatTensor([w["label_ioh_10"]]),
            "label_ioh_15": torch.FloatTensor([w["label_ioh_15"]]),
            "label_nh":     torch.LongTensor([w["label_nh"]]),
            "sqi_end":      torch.FloatTensor([w["sqi_end"]]),
            "caseid":       w["caseid"],
        }


def build_dataloaders(train_ds, val_ds, test_ds, config):
    bs = config["batch_size"]
    kw = dict(num_workers=0, pin_memory=True)
    return (
        DataLoader(train_ds, batch_size=bs, shuffle=True,  drop_last=True, **kw),
        DataLoader(val_ds,   batch_size=bs, shuffle=False, **kw),
        DataLoader(test_ds,  batch_size=bs, shuffle=False, **kw),
    )


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Simplified CNN Encoder
# Input: (B, 2, 3840)  →  output: (B, cnn_out_dim)
# 3 conv blocks:  Conv1d → BN → GELU → MaxPool
# Residual-style: each block adds a 1×1 skip when channels differ
# No WaveNet dilations, no custom causal padding → much more numerically stable
# ─────────────────────────────────────────────────────────────────────────────
class CNNBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel: int = 7) -> None:
        super().__init__()
        pad = kernel // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel, padding=pad)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=pad)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.pool  = nn.MaxPool1d(kernel_size=4, stride=4)
        self.skip  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h  = F.gelu(self.bn1(self.conv1(x)))
        h  = F.gelu(self.bn2(self.conv2(h)))
        h  = h + self.skip(x)   # residual (same length, different channels)
        return self.pool(h)


class SimpleCNNEncoder(nn.Module):
    """3-block CNN encoder: (B, 2, 3840) → (B, cnn_out_dim)"""

    def __init__(self, config: Dict) -> None:
        super().__init__()
        channels   = config["cnn_channels"]      # [32, 64, 128]
        kernel     = config["cnn_kernel_size"]   # 7
        cnn_out    = config["cnn_out_dim"]       # 128

        layers = []
        in_ch  = 2
        for out_ch in channels:
            layers.append(CNNBlock(in_ch, out_ch, kernel))
            in_ch = out_ch
        self.cnn     = nn.Sequential(*layers)
        # Adaptive pool to a fixed number of vectors, then project
        self.gap     = nn.AdaptiveAvgPool1d(1)
        self.proj    = nn.Sequential(
            nn.Linear(in_ch, cnn_out),
            nn.LayerNorm(cnn_out),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 2, 3840)
        h = self.cnn(x)           # (B, 128, T//64)
        h = self.gap(h).squeeze(-1)   # (B, 128)
        return self.proj(h)           # (B, cnn_out_dim)


# ─────────────────────────────────────────────────────────────────────────────
# MLP Fusion: CNN embedding + scalar summary + spectral summary → embedding
# ─────────────────────────────────────────────────────────────────────────────
class MLPFusion(nn.Module):
    def __init__(self, config: Dict) -> None:
        super().__init__()
        in_dim  = (config["cnn_out_dim"] +
                   config["scalar_summary_dim"] +
                   config["spectral_summary_dim"])
        dims    = config["mlp_hidden_dims"]   # [256, 128]
        dropout = config["mlp_dropout"]

        layers = []
        d = in_dim
        for h in dims:
            layers += [nn.Linear(d, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout)]
            d = h
        self.mlp     = nn.Sequential(*layers)
        self.out_dim = d

    def forward(self, cnn_emb, scalar_sum, spectral_sum) -> torch.Tensor:
        x = torch.cat([cnn_emb, scalar_sum, spectral_sum], dim=-1)
        return self.mlp(x)


# ─────────────────────────────────────────────────────────────────────────────
# Full EEGModel
# ─────────────────────────────────────────────────────────────────────────────
class EEGModel(nn.Module):
    """Simplified Model 3: CNN → MLP → IOH heads + N/H head."""

    def __init__(self, config: Dict) -> None:
        super().__init__()
        self.cnn_enc  = SimpleCNNEncoder(config)
        self.fusion   = MLPFusion(config)
        emb_dim       = config["mlp_hidden_dims"][-1]   # 128

        self.ioh_head_5  = nn.Linear(emb_dim, 1)
        self.ioh_head_10 = nn.Linear(emb_dim, 1)
        self.ioh_head_15 = nn.Linear(emb_dim, 1)
        self.nh_head     = nn.Linear(emb_dim, config["nh_n_classes"])

    def encode(self, wav, scalar_sum, spectral_sum):
        # wav: (B, 2, 3840)  — already transposed from dataset
        cnn_emb = self.cnn_enc(wav)
        return self.fusion(cnn_emb, scalar_sum, spectral_sum)

    def forward(self, wav, scalar_sum, spectral_sum) -> Dict[str, torch.Tensor]:
        emb = self.encode(wav, scalar_sum, spectral_sum)
        return {
            "embedding": emb,
            "ioh_5":     self.ioh_head_5(emb),
            "ioh_10":    self.ioh_head_10(emb),
            "ioh_15":    self.ioh_head_15(emb),
            "nh":        self.nh_head(emb),
        }


In [ ]:
def ioh_bce_loss(pred, target, sqi_gate=None):
    bce = F.binary_cross_entropy_with_logits(pred.float(), target.float(), reduction="none")
    if sqi_gate is not None:
        gate    = (sqi_gate >= CONFIG["sqi_train_threshold"]).float().unsqueeze(1)
        n_valid = gate.sum().clamp(min=1.0)
        return (bce * gate).sum() / n_valid
    return bce.mean()


def focal_loss(logits, targets, gamma=2.0, class_weights=None, label_smoothing=0.10):
    """Focal loss with label smoothing, computed in fp32 for stability."""
    n_cls      = logits.size(-1)
    logits_f   = logits.float()
    with torch.no_grad():
        smooth = torch.zeros_like(logits_f).scatter_(1, targets.unsqueeze(1), 1.0)
        smooth = smooth * (1 - label_smoothing) + label_smoothing / n_cls
    log_p = F.log_softmax(logits_f, dim=-1)
    ce    = -(smooth * log_p).sum(dim=-1)
    p     = torch.exp(log_p)
    pt    = (p * F.one_hot(targets, n_cls).float()).sum(dim=-1)
    loss  = (1.0 - pt).pow(gamma) * ce
    if class_weights is not None:
        w    = class_weights.float().to(logits.device).clamp(max=10.0)
        loss = loss * w[targets]
    return loss.clamp(max=100.0).mean()


def compute_class_weights(labels_nh, n_classes=3):
    arr     = np.array(labels_nh, dtype=np.int64)
    counts  = np.maximum(np.bincount(arr, minlength=n_classes).astype(np.float32), 1.0)
    weights = 1.0 / counts
    weights = weights / weights.sum() * n_classes
    return torch.FloatTensor(weights)


In [ ]:
def compute_metrics(ioh_preds, ioh_targets, nh_preds, nh_targets):
    from sklearn.metrics import (roc_auc_score, average_precision_score,
                                  balanced_accuracy_score, f1_score)
    metrics = {}
    for key in ["ioh_5", "ioh_10", "ioh_15"]:
        p = np.nan_to_num(ioh_preds[key], nan=0.5, posinf=1.0, neginf=0.0)
        t = ioh_targets[key]
        if len(np.unique(t)) < 2:
            metrics[f"auroc_{key}"] = float("nan")
            metrics[f"auprc_{key}"] = float("nan")
        else:
            metrics[f"auroc_{key}"] = float(roc_auc_score(t, p))
            metrics[f"auprc_{key}"] = float(average_precision_score(t, p))
    try:
        metrics["nh_balanced_acc"] = float(balanced_accuracy_score(nh_targets, nh_preds))
        metrics["nh_macro_f1"]     = float(f1_score(nh_targets, nh_preds, average="macro", zero_division=0))
    except Exception as e:
        log.warning(f"N/H metrics error: {e}")
        metrics["nh_balanced_acc"] = float("nan")
        metrics["nh_macro_f1"]     = float("nan")
    return metrics


In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, class_weights_nh, config, device):
    model.train()
    total_loss = 0.0; n_batches = 0
    acc_steps  = config["gradient_accumulation_steps"]
    cw         = class_weights_nh.to(device)
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="Train", leave=False)):
        wav         = batch["wav"].to(device).permute(0, 2, 1)   # (B, 2, 3840)
        scalar_sum  = batch["scalar_sum"].to(device)
        spectral_sum = batch["spectral_sum"].to(device)
        sqi_end     = batch["sqi_end"].to(device)
        l5  = batch["label_ioh_5"].to(device)
        l10 = batch["label_ioh_10"].to(device)
        l15 = batch["label_ioh_15"].to(device)
        lnh = batch["label_nh"].squeeze(-1).to(device)

        with torch.amp.autocast(device_type=device.type, enabled=(device.type=="cuda")):
            out = model(wav, scalar_sum, spectral_sum)
            loss_5  = ioh_bce_loss(out["ioh_5"],  l5,  sqi_end)
            loss_10 = ioh_bce_loss(out["ioh_10"], l10, sqi_end)
            loss_15 = ioh_bce_loss(out["ioh_15"], l15, sqi_end)
            loss_nh = focal_loss(out["nh"], lnh, gamma=2.0,
                                  class_weights=cw,
                                  label_smoothing=config["nh_label_smoothing"])
            loss = ((loss_5 + loss_10 + loss_15) / 3.0 + loss_nh) / acc_steps

        loss_val = loss.item() * acc_steps
        if not math.isfinite(loss_val):
            log.warning(f"Step {step}: non-finite loss {loss_val:.4g}, skipping")
            optimizer.zero_grad(); continue

        scaler.scale(loss).backward()
        if (step + 1) % acc_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

        total_loss += loss_val; n_batches += 1

    # flush remaining gradients
    try:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
    except Exception:
        pass

    return total_loss / max(n_batches, 1)


def validate(model, loader, class_weights_nh, config, device):
    model.eval()
    total_loss = 0.0; n_batches = 0
    cw = class_weights_nh.to(device)
    all_preds  = {k: [] for k in ["ioh_5", "ioh_10", "ioh_15"]}
    all_trues  = {k: [] for k in ["ioh_5", "ioh_10", "ioh_15"]}
    all_nh_pred, all_nh_true = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validate", leave=False):
            wav          = batch["wav"].to(device).permute(0, 2, 1)
            scalar_sum   = batch["scalar_sum"].to(device)
            spectral_sum = batch["spectral_sum"].to(device)
            sqi_end      = batch["sqi_end"].to(device)
            l5  = batch["label_ioh_5"].to(device)
            l10 = batch["label_ioh_10"].to(device)
            l15 = batch["label_ioh_15"].to(device)
            lnh = batch["label_nh"].squeeze(-1).to(device)

            out     = model(wav, scalar_sum, spectral_sum)
            loss_5  = ioh_bce_loss(out["ioh_5"],  l5,  sqi_end)
            loss_10 = ioh_bce_loss(out["ioh_10"], l10, sqi_end)
            loss_15 = ioh_bce_loss(out["ioh_15"], l15, sqi_end)
            loss_nh = focal_loss(out["nh"], lnh, gamma=2.0,
                                  class_weights=cw,
                                  label_smoothing=config["nh_label_smoothing"])
            loss = (loss_5 + loss_10 + loss_15) / 3.0 + loss_nh
            total_loss += loss.item(); n_batches += 1

            for key, pred in [("ioh_5", out["ioh_5"]), ("ioh_10", out["ioh_10"]), ("ioh_15", out["ioh_15"])]:
                all_preds[key].append(torch.nan_to_num(torch.sigmoid(pred), nan=0.5).cpu().numpy())
            for key, true in [("ioh_5", l5), ("ioh_10", l10), ("ioh_15", l15)]:
                all_trues[key].append(true.cpu().numpy())
            all_nh_pred.append(torch.nan_to_num(out["nh"], nan=0.0).argmax(-1).cpu().numpy())
            all_nh_true.append(lnh.cpu().numpy())

    val_loss    = total_loss / max(n_batches, 1)
    ioh_preds   = {k: np.concatenate(v).flatten() for k, v in all_preds.items()}
    ioh_targets = {k: np.concatenate(v).flatten() for k, v in all_trues.items()}
    nh_preds    = np.concatenate(all_nh_pred)
    nh_targets  = np.concatenate(all_nh_true)
    metrics     = compute_metrics(ioh_preds, ioh_targets, nh_preds, nh_targets)
    return val_loss, metrics


In [ ]:
class EarlyStopping:
    def __init__(self, patience=8, checkpoint_path="best_model.pt"):
        self.patience        = patience
        self.checkpoint_path = checkpoint_path
        self.best_loss       = float("inf")
        self.counter         = 0
        self.should_stop     = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            log.info(f"  ✓ Best val_loss={val_loss:.6f} — checkpoint saved")
        else:
            self.counter += 1
            log.info(f"  EarlyStopping {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.should_stop = True
                log.info("  Early stopping triggered.")


In [ ]:
def train_model(model, train_loader, val_loader, class_weights_nh, config, device):
    """Single supervised stage — no MSM pretraining needed."""
    model = model.to(device)
    scaler    = torch.amp.GradScaler(device=device.type, enabled=(device.type == "cuda"))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["sup_lr"],
                                   weight_decay=config["weight_decay"])

    warmup = config["warmup_epochs"]
    def lr_lambda(ep):
        if ep < warmup: return (ep + 1) / max(warmup, 1)
        p = (ep - warmup) / max(config["sup_epochs"] - warmup, 1)
        return max(1e-6 / config["sup_lr"], 0.5 * (1.0 + math.cos(math.pi * p)))
    scheduler   = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    early_stop  = EarlyStopping(config["early_stopping_patience"], config["checkpoint_sup"])

    for epoch in range(1, config["sup_epochs"] + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, scaler,
                                   class_weights_nh, config, device)
        val_loss, val_metrics = validate(model, val_loader, class_weights_nh, config, device)
        scheduler.step()
        log.info(
            f"[Ep {epoch:03d}/{config['sup_epochs']}] "
            f"train={tr_loss:.4f}  val={val_loss:.4f}  "
            f"auroc_ioh5={val_metrics.get('auroc_ioh_5', float('nan')):.3f}  "
            f"nh_f1={val_metrics.get('nh_macro_f1', float('nan')):.3f}  "
            f"({time.time()-t0:.1f}s)"
        )
        early_stop.step(val_loss, model)
        if early_stop.should_stop: break

    model.load_state_dict(torch.load(config["checkpoint_sup"], map_location=device,
                                      weights_only=True))
    log.info("Best model loaded from checkpoint.")
    return model


In [ ]:
def evaluate_test(model, test_loader, class_weights_nh, config, device):
    test_loss, metrics = validate(model, test_loader, class_weights_nh, config, device)
    log.info("=== Test Set Evaluation ===")
    log.info(f"  Test loss: {test_loss:.6f}")
    for k, v in metrics.items():
        log.info(f"  {k}: {v:.4f}")
    return metrics


In [ ]:
def plot_predictions(model, test_loader, config, device, n_cases=5):
    model.eval()
    all_pred, all_true, all_cids = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            wav          = batch["wav"].to(device).permute(0, 2, 1)
            scalar_sum   = batch["scalar_sum"].to(device)
            spectral_sum = batch["spectral_sum"].to(device)
            out          = model(wav, scalar_sum, spectral_sum)
            all_pred.extend(torch.sigmoid(out["ioh_5"]).cpu().numpy().flatten().tolist())
            all_true.extend(batch["label_ioh_5"].cpu().numpy().flatten().tolist())
            all_cids.extend(
                batch["caseid"].tolist() if isinstance(batch["caseid"], torch.Tensor)
                else list(batch["caseid"])
            )
            if len(all_pred) >= n_cases * 50: break

    unique_cases = list(dict.fromkeys(all_cids))[:n_cases]
    fig, axes   = plt.subplots(n_cases, 1, figsize=(12, 3 * n_cases), squeeze=False)
    for ax_row, cid in zip(axes, unique_cases):
        ax    = ax_row[0]
        idxs  = [i for i, c in enumerate(all_cids) if c == cid]
        pred_c = [all_pred[i] for i in idxs]
        true_c = [all_true[i] for i in idxs]
        ax.plot(pred_c, label="P(IOH-5min)", color="steelblue")
        ax.fill_between(range(len(true_c)), true_c, alpha=0.3, color="red", label="IOH true")
        ax.set_ylim(0, 1); ax.set_title(f"Case {cid}"); ax.legend(fontsize=8)
    axes[-1][0].set_xlabel("Window index")
    plt.suptitle("Model 3 Simplified — IOH-5 Predictions vs Ground Truth", fontsize=12)
    plt.tight_layout()
    plt.savefig(config["pred_output"], dpi=150); plt.close()
    log.info(f"Prediction plot saved → {config['pred_output']}")


In [ ]:
def main() -> None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"Using device: {device}")

    # ── Clinical metadata ─────────────────────────────────────────────────────
    clinical_df = load_clinical_metadata(CONFIG)

    # ── Case IDs (use BIS monitor cases) ─────────────────────────────────────
    try:
        bis_cases = set(range(1, 21))   # small set for quick testing; expand as needed
    except Exception:
        bis_cases = set(range(1, 6389))
    valid_caseids = sorted(bis_cases)
    log.info(f"Attempting to load {len(valid_caseids)} BIS cases")

    # ── Load waveform data ────────────────────────────────────────────────────
    wav_data, scalar_data = load_waveform_data(valid_caseids, CONFIG)

    # ── Patient-level split ───────────────────────────────────────────────────
    all_loaded = sorted(set(wav_data.keys()) & set(scalar_data.keys()))
    train_caseids, val_caseids, test_caseids = split_cases(
        all_loaded, CONFIG["val_frac"], CONFIG["test_frac"], CONFIG["seed"]
    )

    def _subset(src, keys): return {c: src[c] for c in keys if c in src}

    # ── Window generation ─────────────────────────────────────────────────────
    log.info("Generating training windows …")
    train_windows = generate_windows(_subset(wav_data, train_caseids),
                                      _subset(scalar_data, train_caseids), clinical_df, CONFIG)
    gc.collect()
    log.info("Generating validation windows …")
    val_windows   = generate_windows(_subset(wav_data, val_caseids),
                                      _subset(scalar_data, val_caseids), clinical_df, CONFIG)
    gc.collect()
    log.info("Generating test windows …")
    test_windows  = generate_windows(_subset(wav_data, test_caseids),
                                      _subset(scalar_data, test_caseids), clinical_df, CONFIG)
    del wav_data, scalar_data; gc.collect()

    if not train_windows:
        log.error("No training windows generated — aborting"); return

    # ── Scalers ───────────────────────────────────────────────────────────────
    sc_scalar, sc_spectral, eeg_mean, eeg_std = fit_scalers(train_windows)
    train_windows = apply_scalers(train_windows, sc_scalar, sc_spectral, eeg_mean, eeg_std)
    val_windows   = apply_scalers(val_windows,   sc_scalar, sc_spectral, eeg_mean, eeg_std)
    test_windows  = apply_scalers(test_windows,  sc_scalar, sc_spectral, eeg_mean, eeg_std)

    # ── Class weights ─────────────────────────────────────────────────────────
    class_weights_nh = compute_class_weights(
        [w["label_nh"] for w in train_windows], CONFIG["nh_n_classes"]
    )
    log.info(f"N/H class weights: {class_weights_nh.tolist()}")

    # ── Datasets & DataLoaders ────────────────────────────────────────────────
    train_ds = EEGDataset(train_windows)
    val_ds   = EEGDataset(val_windows)
    test_ds  = EEGDataset(test_windows)
    train_loader, val_loader, test_loader = build_dataloaders(train_ds, val_ds, test_ds, CONFIG)

    # ── Model ─────────────────────────────────────────────────────────────────
    model    = EEGModel(CONFIG)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"EEGModel (simplified) parameters: {n_params:,}")

    # ── Train ─────────────────────────────────────────────────────────────────
    model = train_model(model, train_loader, val_loader, class_weights_nh, CONFIG, device)

    # ── Evaluate ──────────────────────────────────────────────────────────────
    evaluate_test(model, test_loader, class_weights_nh, CONFIG, device)

    # ── Plot predictions ──────────────────────────────────────────────────────
    plot_predictions(model, test_loader, CONFIG, device, n_cases=5)

    log.info("model_3_simple pipeline complete.")


if __name__ == "__main__":
    main()


In [ ]:
main()